<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Test File for Depth Camera (D435i / D405)
  </span>
</div>

<span style="color: magenta; font-weight: bold; font-size: 30px;">
Coordinate Information
</span>

<div style="color: white; font-size: 15px; line-height: 1.6;">

  <b>Image / Depth Coordinate Convention</b>

  <table style="border-collapse: collapse; margin-top: 5px; color: white;">
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>Pixel</b></td>
      <td><code>(u, v)</code></td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>Pixel origin</b></td>
      <td>Top-left corner of the image (= Camera optical center)</td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>u-axis</b></td>
      <td>Left → Right</td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>v-axis</b></td>
      <td>Top → Bottom</td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>Depth</b></td>
      <td><code>z</code></td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>3D position</b></td>
      <td><code>(X, Y, Z)</code></td>
    </tr>
    <tr>
      <td style="padding: 4px 14px 4px 2;"><b>3D axes</b></td>
      <td>
        <code>X</code>: Left → Right,
        <code>Y</code>: Top → Bottom,
        <code>Z</code>: Depth direction
      </td>
    </tr>
  </table>

</div>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:gray; font-weight:bold; font-size:50px;">
Check recognition of D435i and D405
</span>

In [2]:
ctx = rs.context()
devices = ctx.query_devices()

print("Number of RealSense devices:", len(devices))

for i, dev in enumerate(devices):
    name = dev.get_info(rs.camera_info.name)
    serial = dev.get_info(rs.camera_info.serial_number)
    product_line = dev.get_info(rs.camera_info.product_line)
    firmware = dev.get_info(rs.camera_info.firmware_version)
    print(f"[{i}] Name={name}, Serial={serial}, Product Line={product_line}, Firmware={firmware}")

Number of RealSense devices: 3
[0] Name=Intel RealSense D405, Serial=260322279815, Product Line=D400, Firmware=5.15.1.55
[1] Name=Intel RealSense D435I, Serial=250122070658, Product Line=D400, Firmware=5.15.1.55
[2] Name=Intel RealSense D405, Serial=260422275816, Product Line=D400, Firmware=5.15.1.55


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Robot
</span>

In [3]:
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot
robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="color:gray; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [6]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="color:gray; font-weight:bold; font-size:50px;">
Posture Change
</span>

In [5]:
Desired_Pos = ysw.DEMO_BASIC_POS_260826     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization of Vision
</span>

In [2]:
d435i = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.HEAD_D435I_SERIAL)
d435i.initialize()

d405_R = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.RIGHT_D405_SERIAL)
d405_R.initialize()

d405_L = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.LEFT_D405_SERIAL)
d405_L.initialize()

<span style="color:green; font-weight:bold; font-size:50px;">
Show ALL Cameras [One D435i / Two D405]
</span> 

In [5]:
"""
# Simple test of 3 RGB-D cameras
# 2 x 3 view
# ESC: Terminate

┌────────────────┬──────────────┬────────────────┐
│ L wrist color  │ Head color   │ R wrist color  │
├────────────────┼──────────────┼────────────────┤
│ L wrist depth  │ Head depth   │ R wrist depth  │
└────────────────┴──────────────┴────────────────┘
"""

camera_dict = {
    # 순서 중요! 위 표와 동일한 순서로
    "wrist_L": d405_L,
    "head": d435i,
    "wrist_R": d405_R,
}

window_name = "RGB-D Cameras: Left Wrist / Head / Right Wrist"

# Display panel size
panel_w = 640
panel_h = 480

def make_blank_panel(text, width=640, height=480):
    image = np.zeros((height, width, 3), dtype=np.uint8)
    cv2.putText(
        image,
        text,
        (30, height // 2),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )
    return image


def resize_and_label(image, label, width=640, height=480):
    if image is None:
        image = make_blank_panel("No Image", width, height)
    else:
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

        image = cv2.resize(image, (width, height))

    cv2.putText(
        image,
        label,
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2
    )

    return image



# Start all cameras
started_cameras = []

try:
    for name, cam in camera_dict.items():
        cam.startPipeline()
        started_cameras.append(cam)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    # Fullscreen mode
    cv2.setWindowProperty(
        window_name,
        cv2.WND_PROP_FULLSCREEN,
        cv2.WINDOW_FULLSCREEN
    )

    while True:
        color_panels = []
        depth_panels = []

        for name, cam in camera_dict.items():
            color_frame, depth_frame = cam.getRawFrames()

            if color_frame is None and depth_frame is None:
                color_panel = make_blank_panel(f"{name}: No Frame", panel_w, panel_h)
                depth_panel = make_blank_panel(f"{name}: No Frame", panel_w, panel_h)

            else:
                color_image, depth_image = cam.frameToImage(color_frame, depth_frame)

                if color_image is not None:
                    color_panel = resize_and_label(
                        color_image,
                        f"{name} color",
                        panel_w,
                        panel_h
                    )
                else:
                    color_panel = make_blank_panel(
                        f"{name}: No Color",
                        panel_w,
                        panel_h
                    )

                if depth_image is not None:
                    depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)
                    depth_panel = resize_and_label(
                        depth_colorMap,
                        f"{name} depth",
                        panel_w,
                        panel_h
                    )
                else:
                    depth_panel = make_blank_panel(
                        f"{name}: No Depth",
                        panel_w,
                        panel_h
                    )

            color_panels.append(color_panel)
            depth_panels.append(depth_panel)

        top_row = cv2.hconcat(color_panels)
        bottom_row = cv2.hconcat(depth_panels)

        grid_image = cv2.vconcat([top_row, bottom_row])

        cv2.imshow(window_name, grid_image)

        key = cv2.waitKey(1)

        if key == 27:       # ESC key
            break

finally:
    for name, cam in camera_dict.items():
        cam.stopPipeline()
        cam.destroyAllWindows()
    

[INFO] [Left Wrist D405] Pipeline started.
[INFO] [Head D435i] Pipeline started.
[INFO] [Right Wrist D405] Pipeline started.


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[INFO] [Left Wrist D405] Pipeline stopped.
[INFO] [Head D435i] Pipeline stopped.
[INFO] [Right Wrist D405] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Show individual camera [D435i / D405]
</span>

In [ ]:
# Simple test of RGB-D camera
# Click ESC: Terminate

# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L
    
rv = cam.startPipeline()

# if rv is False:
#     print(f"[WARNING] Camera[{location}] start failed. Try recovery...")
#     rv = cam.recoverPipeline(
#         align_enable=True,
#         try_hardware_reset=True,
#         wait_after_reset_sec=5.0,
#     )
# if rv is False:
#     raise RuntimeError(f"Camera[{location}] Recovery failed.")    


try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()
        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)
        
        if (cam.rgb_on):
            cam.imshow("color_image", color_image)
        if (cam.depth_on):
            cam.imshow("depth_image", depth_colorMap)
        
        key = cam.waitKey(1)
        if key == 27:       # ESC key
            break
        
finally:
    cam.stopPipeline()
    cam.destroyAllWindows()
        

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Distance of center pixel
</span>

In [9]:
# Get distance value of center pixel of image
# Click ESC: Terminate

# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L
    
cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()
        
        if color_frame is None or depth_frame is None:
            continue
        
        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top

        distance = cam.getDepth(depth_frame, (u, v))   # unit = [m]

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)

        key = cam.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.


[INFO] [Head D435i] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of center pixel based on Camera Reference Frame
</span>

In [ ]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

# Show center 3D point position [x,y,z] in camera frame
cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        h, w, _ = color_image.shape
        u = w // 2      # origin is left    
        v = h // 2      # origin is top 

        pos = cam.get3DPos(depth_frame, (u,v))
        X, Y, Z = pos

        cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,
            f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)

        key = cam.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Right Wrist D405] Pipeline started.
[INFO] [Right Wrist D405] Pipeline stopped.


<span style="color:#B8860B; font-weight:bold; font-size:50px;">
3D position of clicked point 
</span>

In [12]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

cam.openWindowWithMouse("color_image")

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        if cam.clicked_point is not None:
            u, v = cam.clicked_point

            X, Y, Z = cam.get3DPos(depth_frame, (u,v))

            cv2.circle(color_image, (u, v), 5, (0, 0, 255), -1)
            cv2.putText(
                color_image,
                f"X:{X:.3f}, Y:{Y:.3f}, Z:{Z:.3f} m",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            print(f"Pixel=({u}, {v}), Camera 3D=({X:.3f}, {Y:.3f}, {Z:.3f}) m")

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.
Clicked pixel: u=407, v=232
Pixel=(407, 232), Camera 3D=(0.546, -0.080, 3.384) m
Pixel=(407, 232), Camera 3D=(0.549, -0.080, 3.403) m
Pixel=(407, 232), Camera 3D=(0.546, -0.080, 3.384) m
Pixel=(407, 232), Camera 3D=(0.552, -0.081, 3.422) m
Pixel=(407, 232), Camera 3D=(0.534, -0.078, 3.312) m
Pixel=(407, 232), Camera 3D=(0.534, -0.078, 3.312) m
Pixel=(407, 232), Camera 3D=(0.540, -0.079, 3.348) m
Pixel=(407, 232), Camera 3D=(0.552, -0.081, 3.422) m
Pixel=(407, 232), Camera 3D=(0.523, -0.077, 3.242) m
Pixel=(407, 232), Camera 3D=(0.534, -0.078, 3.312) m
Pixel=(407, 232), Camera 3D=(0.540, -0.079, 3.348) m
Pixel=(407, 232), Camera 3D=(0.531, -0.078, 3.294) m
Pixel=(407, 232), Camera 3D=(0.520, -0.076, 3.225) m
Pixel=(407, 232), Camera 3D=(0.543, -0.079, 3.366) m
Pixel=(407, 232), Camera 3D=(0.549, -0.080, 3.403) m
Pixel=(407, 232), Camera 3D=(0.531, -0.078, 3.294) m
Pixel=(407, 232), Camera 3D=(0.549, -0.080, 3.403) m
Pixel=(407, 232), Camera 3D=(0.53

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object (2D BBox)
</span>

In [7]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = cam.detectColorObjectContour(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"pixel={obj['pixel']}, "
                f"pos={np.round(obj['pos'], 3)}, "
                f"area={obj['area']:.1f}"
            )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)
        cam.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.
[object 0] pixel=[320, 468], pos=[0.011 0.212 0.581], area=8939.0
[object 1] pixel=[344, 363], pos=[0.037 0.124 0.645], area=301.5
[object 2] pixel=[481, 363], pos=[0.183 0.124 0.646], area=366.0
[object 6] pixel=[532, 339], pos=[0.241 0.1   0.657], area=1244.5
[object 18] pixel=[557, 295], pos=[0.299 0.059 0.732], area=429.5
[object 22] pixel=[285, 257], pos=[-0.03   0.013  0.768], area=338.0
[object 23] pixel=[479, 270], pos=[0.211 0.029 0.754], area=1950.0
[object 24] pixel=[447, 248], pos=[0.179 0.002 0.789], area=337.0
[object 25] pixel=[394, 250], pos=[0.109 0.005 0.782], area=698.0
[object 31] pixel=[405, 131], pos=[ 0.466 -0.559  2.947], area=690.5
[object 33] pixel=[413, 83], pos=[ 0.541 -0.849  3.159], area=320.0
[object 35] pixel=[322, 37], pos=[ 0.078 -1.248  3.623], area=21892.5
[object 0] pixel=[320, 468], pos=[0.011 0.212 0.581], area=7604.0
[object 2] pixel=[478, 363], pos=[0.18  0.124 0.646], area=310.5
[object 3] pixel=[532, 339],

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center 3D position of Colored Object (3D BBox)
</span>

In [3]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = cam.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_ORANGE,       # Can be changed    
            minimum_area=50                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}, "
                f"area_2d={np.round(obj['area_2d'], 3)}"
            )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)
        cam.imshow("mask", mask)

        key = cam.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.
[object 0] center_3d=[0.128 0.167 0.664], bbox_3d_size=[0.02  0.006 0.003], area_2d=81.5
[object 1] center_3d=[-0.086  0.165  0.664], bbox_3d_size=[0.031 0.028 0.053], area_2d=803.0
[object 2] center_3d=[0.005 0.161 0.661], bbox_3d_size=[0.024 0.03  0.054], area_2d=750.0
[object 3] center_3d=[0.012 0.122 0.704], bbox_3d_size=[0.024 0.028 0.054], area_2d=632.0
[object 4] center_3d=[-0.076  0.117  0.704], bbox_3d_size=[0.031 0.029 0.067], area_2d=691.5
[object 5] center_3d=[0.134 0.083 0.749], bbox_3d_size=[0.708 0.201 2.644], area_2d=15115.5
[object 6] center_3d=[-0.359 -0.062  0.915], bbox_3d_size=[0.029 0.026 0.032], area_2d=298.5
[object 7] center_3d=[-0.012 -0.069  0.925], bbox_3d_size=[0.023 0.019 0.013], area_2d=361.5


QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

[object 0] center_3d=[0.003 0.168 0.648], bbox_3d_size=[0.022 0.017 0.006], area_2d=267.0
[object 1] center_3d=[0.126 0.166 0.663], bbox_3d_size=[0.093 0.018 0.009], area_2d=608.0
[object 2] center_3d=[-0.085  0.163  0.65 ], bbox_3d_size=[0.026 0.032 0.034], area_2d=736.0
[object 3] center_3d=[0.012 0.124 0.699], bbox_3d_size=[0.019 0.019 0.005], area_2d=248.0
[object 4] center_3d=[-0.076  0.117  0.706], bbox_3d_size=[0.026 0.033 0.027], area_2d=642.0
[object 5] center_3d=[-0.011 -0.075  0.929], bbox_3d_size=[0.025 0.04  0.032], area_2d=352.0
[object 0] center_3d=[0.005 0.169 0.651], bbox_3d_size=[0.024 0.015 0.007], area_2d=273.5
[object 1] center_3d=[0.086 0.163 0.66 ], bbox_3d_size=[0.016 0.01  0.006], area_2d=71.5
[object 2] center_3d=[0.136 0.165 0.662], bbox_3d_size=[0.068 0.018 0.01 ], area_2d=511.0
[object 3] center_3d=[-0.085  0.163  0.651], bbox_3d_size=[0.026 0.032 0.035], area_2d=761.0
[object 4] center_3d=[0.013 0.126 0.702], bbox_3d_size=[0.02  0.016 0.008], area_2d=206.0

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Center Depth of ROI with ROI Box
</span>

In [1]:
# --- Select camera location ----------------------------
location = "head"       # head / wrist_R / wrist_L
# -------------------------------------------------------

if location == "head":
    cam = d435i
elif location == "wrist_R":
    cam = d405_R
elif location == "wrist_L":
    cam = d405_L
    

# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: PACK_POS
# ============================================================
width = 640
height = 480
pixel_roi_x = [140, width - 140]      
pixel_roi_y = [320, height]    
# pixel_roi_x = None            # None이면 x 전체 사용
# pixel_roi_y = None            # None이면 y 전체 사용
    
    
# ============================================================
# Get the Distance of Center Pixel 
# Table distance is approximated as the depth of center pixel
# ============================================================

cam.startPipeline()

try:
    while True:
        color_frame, depth_frame = cam.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
        depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)
        
        h, w, _ = color_image.shape
        
    
        # ROI center pixel in full image coordinate
        if pixel_roi_x is None:
            x_min = 0
            x_max = w
        else:
            x_min, x_max = pixel_roi_x
            x_min = int(np.clip(x_min, 0, w - 1))
            x_max = int(np.clip(x_max, 0, w))

        if pixel_roi_y is None:
            y_min = 0
            y_max = h
        else:
            y_min, y_max = pixel_roi_y
            y_min = int(np.clip(y_min, 0, h - 1))
            y_max = int(np.clip(y_max, 0, h))
            
        x_roi_center = (x_min + x_max) // 2
        y_roi_center = (y_min + y_max) // 2

        distance = d435i.getDepth(depth_frame, (x_roi_center, y_roi_center))   # unit = [m]
        
        
        # Draw ROI box on full image
        cv2.rectangle(
            color_image,
            (x_min, y_min),
            (x_max, y_max),
            (255, 0, 0),
            2
        )
        cv2.putText(
            color_image,                   
            f"ROI Box",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )
        
        # Draw ROI center point on full image
        cv2.circle(color_image, (x_roi_center, y_roi_center), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"[ROI center point] Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        cam.imshow("color_image", color_image)
        cam.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    cam.stopPipeline()
    cam.destroyAllWindows()

NameError: name 'd435i' is not defined